# Fine-tune Llama 3.1 8B with Unsloth

This notebook demonstrates how to fine-tune the Llama 3.1 8B model using the [Unsloth](https://github.com/unslothai/unsloth) library. We will use a custom dataset (Bhagavad Gita Q&A) and export the final model to GGUF format for use with Ollama.

**Note**: This notebook is designed to run on a free Google Colab instance (T4 GPU).

## 1. Install Dependencies

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## 2. Load Model and Tokenizer
We use the 4-bit quantized version of Llama 3.1 8B to fit within Colab's memory limits.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 405b model 4bit. NB: 405b requires usually 8xH100s for finetuning.
    "unsloth/Llama-3.2-1B-bnb-4bit",           # Llama 3.2 models
    "unsloth/Llama-3.2-3B-bnb-4bit",
]
# We will use the Instruct version since we are doing Chat Fine-Tuning
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like navigation-llama
)

## 3. Setup LoRA Adapters
We need to add LoRA adapters to the model to enable efficient fine-tuning.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

## 4. Load and Format Dataset
We will load the `gita_qna_for_finetune.jsonl` file. 
> **IMPORTANT**: Upload your `gita_qna_for_finetune.jsonl` file to the Colab files section or mount your Google Drive before running this cell.

In [ ]:
from datasets import load_dataset

# Check if file exists, if not, printing instructions
import os
dataset_path = "gita_qna_for_finetune.jsonl"

if not os.path.exists(dataset_path):
    print(f"WARNING: {dataset_path} not found. Please upload it to the current directory.")
    # You can also search in drive if mounted
    # dataset_path = "/content/drive/MyDrive/path/to/gita_qna_for_finetune.jsonl"

dataset = load_dataset("json", data_files = dataset_path, split = "train")

# Apply standard Llama 3 chat template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

dataset = dataset.map(formatting_prompts_func, batched = True)

## 5. Train the Model
We use the `SFTTrainer` from TRL to fine-tune the model.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase this for full training (e.g., 100-500)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()

## 6. Inference
Let's test the model before saving it.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "system", "content": "You are a wise teacher drawing from Bhagavad Gita."},
    {"role": "user", "content": "Using Chapter 2, Verse 47, answer: What is the nature of duty?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

## 7. Saving and GGUF Export
We will now save the model and export it to GGUF format so it can be used with Ollama.

### Save LoRA Adapters

In [ ]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")

### Export to GGUF
This will convert the model to GGUF format. `quantization_method` can be `q4_k_m`, `q5_k_m`, `q8_0` or `f16`.
The file will be saved as `model.gguf` (or similar depending on quantization) in the current directory.

Once this is done, you can download the `.gguf` file.

In [ ]:
# Save to GGUF
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")

## 8. Usage with Ollama
1. Download the generated `model-unsloth.Q4_K_M.gguf` file to your local machine.
2. Create a `Modelfile`:
   ```dockerfile
   FROM ./model-unsloth.Q4_K_M.gguf
   TEMPLATE """{{ if .System }}<|start_header_id|>system<|end_header_id|>\n\n{{ .System }}<|eot_id|>{{ end }}{{ if .Prompt }}<|start_header_id|>user<|end_header_id|>\n\n{{ .Prompt }}<|eot_id|>{{ end }}<|start_header_id|>assistant<|end_header_id|>\n\n"""
   PARAMETER stop "<|start_header_id|>"
   PARAMETER stop "<|end_header_id|>"
   PARAMETER stop "<|eot_id|>"
   ```
3. Run `ollama create gita-llama -f Modelfile`
4. Run `ollama run gita-llama`